# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [ ]:
!pip install -q --upgrade pypdf gradio openai python-dotenv


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(document_text[:500])


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if not os.getenv("NVIDIA_API_KEY"):
    print("Por favor, configura NVIDIA_API_KEY en tu archivo .env")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [ ]:
# ==============================================================================
# Paso 3: Función de chat con el documento (Implementado por Nandez)
# ==============================================================================

def build_system_prompt(document_text: str) -> str:
    """
    Construye el system prompt que define el rol del asistente y contiene
    el documento como contexto único de verdad, instruyendo al modelo a
    limitar sus respuestas estrictamente al texto proporcionado.
    """
    system_prompt = f"""Eres un asistente de investigación experto y riguroso en el artículo científico:
"Attention Is All You Need" (Vaswani et al., 2017).

Tu única misión es responder preguntas sobre el documento adjunto de forma técnica, precisa y veraz.

Directrices estrictas:
1. Responde ÚNICAMENTE basándote en la información explícita del documento adjunto a continuación.
2. Si la respuesta no se encuentra en el documento (por ejemplo, preguntas sobre modelos posteriores como GPT-4, conceptos ajenos al paper o cultura general), debes responder exactamente:
   "No encuentro esa información en el documento."
3. No inventes, no supongas ni utilices conocimientos externos no incluidos en el texto.
4. Mantén un tono académico, formal y conciso.

=== INICIO DEL DOCUMENTO ===
{document_text}
=== FIN DEL DOCUMENTO ===
"""
    return system_prompt


def history_to_messages(history):
    """
    Convierte el historial de Gradio al formato de mensajes de OpenAI
    con roles 'user' y 'assistant'.
    
    Compatible con listas de dicts [{'role': ..., 'content': ...}] y
    con listas tradicionales de pares/tuplas [[user, assistant], ...].
    """
    messages = []
    if not history:
        return messages

    # Formato de Gradio con mensajes estructurados (dicts)
    if isinstance(history, list) and len(history) > 0 and isinstance(history[0], dict):
        for msg in history:
            role = msg.get("role")
            content = msg.get("content")
            if role in ("user", "assistant") and content:
                messages.append({"role": role, "content": content})
        return messages

    # Formato tradicional de tuplas/listas [user_msg, assistant_msg]
    for turn in history:
        if isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": str(user_msg)})
            if bot_msg:
                messages.append({"role": "assistant", "content": str(bot_msg)})

    return messages


def chat_con_documento(message, history, document_text):
    """
    Función generadora para Gradio que gestiona la conversación con streaming,
    inyectando el system prompt con el documento y el historial acumulado.
    """
    # 1. Iniciar con el system prompt delimitado
    system_prompt = build_system_prompt(document_text)
    messages = [{"role": "system", "content": system_prompt}]

    # 2. Agregar el historial previo de conversación
    messages.extend(history_to_messages(history))

    # 3. Agregar el nuevo mensaje del usuario
    messages.append({"role": "user", "content": message})

    # 4. Llamar a la API de NVIDIA NIM con streaming habilitado
    try:
        response_stream = client.chat.completions.create(
            model=MODELO,
            messages=messages,
            temperature=1.0,
            top_p=0.95,
            stream=True
        )

        # 5. Acumular y entregar tokens en tiempo real con yield
        respuesta_acumulada = ""
        for chunk in response_stream:
            if chunk.choices and len(chunk.choices) > 0:
                delta = chunk.choices[0].delta.content
                if delta:
                    respuesta_acumulada += delta
                    yield respuesta_acumulada

    except Exception as e:
        yield f"Error al procesar la solicitud con NVIDIA NIM: {str(e)}"


## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [ ]:
# ==============================================================================
# Paso 4: Interfaz de usuario con Gradio (Implementado por Nandez)
# ==============================================================================

import gradio as gr
from pypdf import PdfReader

# 1. Extraer el texto del PDF usando la función del Paso 1
pdf_path = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

# 2. Calcular y mostrar métricas de extracción
reader = PdfReader(pdf_path)
total_paginas = len(reader.pages)
total_caracteres = len(document_text)

print(f"Extracción del documento completada exitosamente:")
print(f"  • Archivo procesado: {pdf_path}")
print(f"  • Total de páginas: {total_paginas}")
print(f"  • Total de caracteres: {total_caracteres:,}")

# 3. Preguntas de ejemplo sobre el paper
# Gradio exige que cada ejemplo sea una lista [pregunta, document_text]
preguntas_base = [
    "¿Cuál es la arquitectura principal propuesta en el paper y cuáles son sus partes?",
    "¿Cómo funciona el mecanismo de Scaled Dot-Product Attention?",
    "¿Cuántas capas componen el encoder y el decoder en el modelo base?",
    "¿Por qué los autores deciden prescindir de recurrencia (RNN) y convoluciones (CNN)?",
    "¿Qué es GPT-4?",  # Pregunta trampa / control (no está en el paper)
]

ejemplos_formateados = [[pregunta, document_text] for pregunta in preguntas_base]

# 4. Configuración de la interfaz con gr.ChatInterface
demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente de Investigación: Attention Is All You Need",
    description=(
        "Asistente conversacional con recuperación sobre contexto completo del paper "
        "*(Vaswani et al., 2017)* impulsado por **NVIDIA NIM** en streaming. "
        "El modelo únicamente responde preguntas con sustento en el documento; "
        "si se le pregunta algo externo, indicará que no lo encuentra."
    ),
    additional_inputs=[
        gr.Textbox(
            value=document_text,
            visible=False,
            label="Document Context",
        )
    ],
    examples=ejemplos_formateados,
    textbox=gr.Textbox(
        placeholder="Haz una pregunta sobre la arquitectura Transformer...",
        container=False,
        scale=7,
    ),
)

# 5. Lanzar la interfaz web interactiva
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)